# 第 5 章:预训练(见证奇迹的一章)

前 4 章的积木终于要动起来:让模型真的从数据里学出语言规律,
生成出不是乱码的文本。

**章节内容**
1. 准备语料(tinyshakespeare,约 1MB,几分钟就能看到 loss 下降)
2. 训练循环:CrossEntropyLoss + AdamW + 梯度裁剪 + 混合精度
3. 周期性评估 train/val loss + 生成样本
4. checkpoint 保存/加载
5. 采样策略:贪心 → 温度 → top-k → top-p
6. 封装到 `llm/train.py` + `llm/generate.py`

**硬件预期**(GTX 1060 6GB)
- 小配置(emb=128, layers=4, ctx=128)约 3-5M 参数,batch=32 单卡可跑
- 约 30 分钟能看到明显的文本结构,1-2 小时生成基本通顺的莎士比亚风格

In [1]:
import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('..').resolve()))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import time
import torch
import torch.nn as nn

from llm.tokenizer import Tokenizer
from llm.dataset   import make_dataloader
from llm.model     import GPTModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
torch.manual_seed(123)

device: cuda


---
## 1. 准备语料

下载 `tinyshakespeare`(约 1.1 MB),Karpathy 的经典小语料。
如果下载失败,可以把任意一段英文文本存成 `../data/tinyshakespeare.txt` 手动使用。

In [2]:
import urllib.request

data_path = pathlib.Path('../data/tinyshakespeare.txt')
if not data_path.exists():
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    data_path.parent.mkdir(exist_ok=True)
    print('下载中 ...')
    urllib.request.urlretrieve(url, data_path)

raw_text = data_path.read_text(encoding='utf-8')
print(f'语料长度: {len(raw_text):,} 字符')
print('前 200 字符:')
print(raw_text[:200])

下载中 ...
语料长度: 1,115,394 字符
前 200 字符:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [3]:
tokenizer = Tokenizer()
token_ids = tokenizer.encode(raw_text)
print(f'token 总数: {len(token_ids):,}')

# 9:1 切分 train/val
split = int(0.9 * len(token_ids))
train_ids = token_ids[:split]
val_ids   = token_ids[split:]
print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

token 总数: 338,025
train: 304,222   val: 33,803


---
## 2. 小模型配置 + DataLoader

In [4]:
CFG = dict(
    vocab_size=tokenizer.vocab_size,
    context_length=128,
    emb_dim=128,
    n_heads=4,
    n_layers=4,
    drop_rate=0.1,
    qkv_bias=False,
)
BATCH_SIZE = 32
STRIDE     = CFG['context_length']   # 窗口不重叠

train_loader = make_dataloader(train_ids, CFG['context_length'], STRIDE, BATCH_SIZE, shuffle=True)
val_loader   = make_dataloader(val_ids,   CFG['context_length'], STRIDE, BATCH_SIZE, shuffle=False)

print(f'train batches: {len(train_loader)}   val batches: {len(val_loader)}')

model = GPTModel(CFG).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'参数量: {n_params/1e6:.2f} M')

train batches: 74   val batches: 8
参数量: 13.67 M


---
## 3. 损失函数 + 评估

`nn.CrossEntropyLoss` 要求:
- logits shape: `(N, C)` 或 `(N, C, ...)` —— C 是类别数
- target shape: `(N,)` 或 `(N, ...)` —— 整数类别 id

我们模型输出是 `(B, T, V)`,目标是 `(B, T)`,把前两维展平送进去:

In [ ]:
# TODO: 实现 calc_loss_batch —— 单 batch 的 loss

def calc_loss_batch(x, y, model, device):
    x, y = x.to(device), y.to(device)
    logits = model(x)  # (B, T, V)
    # TODO: flatten 后用 F.cross_entropy
    # return torch.nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())
    ...


def calc_loss_loader(loader, model, device, num_batches=None):
    """在给定 loader 上计算平均 loss(取 num_batches 个 batch)。"""
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if num_batches is not None and i >= num_batches:
                break
            total += calc_loss_batch(x, y, model, device).item()
            n += 1
    model.train()
    return total / max(n, 1)


# 训练前快速看一下初始 loss(未训练时应接近 ln(vocab_size) ≈ 10.8)
import math
print(f'未训练 train loss: {calc_loss_loader(train_loader, model, device, num_batches=5):.3f}')
print(f'理论初始值 ln(V):   {math.log(CFG["vocab_size"]):.3f}')

---
## 4. 采样函数(先写好,训练中用来观察效果)

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, context_length,
             temperature=1.0, top_k=None, top_p=None):
    """
    温度 + top-k + top-p 采样。
    - temperature=1.0 : 标准 softmax
    - temperature->0  : 趋近 argmax(贪心)
    - temperature>1   : 更随机
    - top_k=50        : 只从概率最高的 50 个候选里采样
    - top_p=0.9       : 只从累积概率 >= 0.9 的最小候选集里采样
    """
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]
        logits = model(idx_cond)[:, -1, :]   # (B, V)

        # 温度
        logits = logits / max(temperature, 1e-8)

        # top-k
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, -1:]] = float('-inf')

        probs = torch.softmax(logits, dim=-1)

        # top-p (nucleus)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum = torch.cumsum(sorted_probs, dim=-1)
            mask = cum > top_p
            # 保留第一个超过 top_p 的,其余置 0
            mask[..., 0] = False
            sorted_probs[mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(-1, keepdim=True)
            next_id_local = torch.multinomial(sorted_probs, 1)
            next_id = sorted_idx.gather(-1, next_id_local)
        else:
            next_id = torch.multinomial(probs, 1)

        idx = torch.cat([idx, next_id], dim=1)
    return idx


def sample_text(model, tokenizer, prompt, max_new_tokens=60, **kw):
    ids = torch.tensor([tokenizer.encode(prompt)], device=device)
    out = generate(model, ids, max_new_tokens, CFG['context_length'], **kw)
    return tokenizer.decode(out[0].tolist())

print('未训练样本:')
print(sample_text(model, tokenizer, 'ROMEO:', max_new_tokens=30, temperature=1.0, top_k=40))

---
## 5. 训练循环(含梯度裁剪 + 混合精度)

GTX 1060 显存不大,开启混合精度能省一半显存、略微加速。

In [ ]:
# TODO: 补全训练循环的核心五步

def train(model, train_loader, val_loader, optimizer,
          num_epochs, eval_every=100, eval_batches=20, use_amp=True):
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == 'cuda'))
    history = {'step': [], 'train_loss': [], 'val_loss': []}
    step = 0
    t0 = time.time()

    for epoch in range(num_epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            # TODO 核心五步
            # optimizer.zero_grad()
            # with torch.cuda.amp.autocast(enabled=(use_amp and device=='cuda')):
            #     logits = model(x)
            #     loss = F.cross_entropy(logits.flatten(0,1), y.flatten())
            # scaler.scale(loss).backward()
            # scaler.unscale_(optimizer)
            # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # scaler.step(optimizer)
            # scaler.update()
            ...

            step += 1
            if step % eval_every == 0:
                tr = calc_loss_loader(train_loader, model, device, eval_batches)
                vl = calc_loss_loader(val_loader,   model, device, eval_batches)
                history['step'].append(step)
                history['train_loss'].append(tr)
                history['val_loss'].append(vl)
                dt = time.time() - t0
                print(f'[epoch {epoch+1} step {step:5d} ({dt:.0f}s)] '
                      f'train={tr:.3f}  val={vl:.3f}')
                # 顺便打印一个生成样本
                print('  > ' + sample_text(model, tokenizer, 'ROMEO:',
                                           max_new_tokens=40, temperature=0.8, top_k=40)
                      .replace('\n', ' / ')[:120])
    return history

# 训练!
import torch.nn.functional as F
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
history = train(model, train_loader, val_loader, optimizer,
                num_epochs=1, eval_every=50, eval_batches=10)

In [ ]:
# 画 loss 曲线
import matplotlib.pyplot as plt
plt.plot(history['step'], history['train_loss'], label='train')
plt.plot(history['step'], history['val_loss'],   label='val')
plt.xlabel('step'); plt.ylabel('loss'); plt.grid(True); plt.legend(); plt.show()

---
## 6. 保存 / 加载 checkpoint

In [ ]:
ckpt_path = pathlib.Path('../checkpoints/gpt_small.pt')
ckpt_path.parent.mkdir(exist_ok=True)

torch.save({
    'model':     model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'cfg':       CFG,
}, ckpt_path)
print(f'已保存: {ckpt_path}  ({ckpt_path.stat().st_size/1e6:.1f} MB)')

# 加载演示
state = torch.load(ckpt_path, map_location=device)
model2 = GPTModel(state['cfg']).to(device)
model2.load_state_dict(state['model'])
print('加载成功')

---
## 7. 对比采样策略

In [ ]:
prompt = 'ROMEO:'
print('=== 贪心(temperature→0) ===')
print(sample_text(model, tokenizer, prompt, 80, temperature=0.01))
print('\n=== 温度 0.8 + top-k 40 ===')
print(sample_text(model, tokenizer, prompt, 80, temperature=0.8, top_k=40))
print('\n=== 温度 1.2 + top-p 0.9 ===')
print(sample_text(model, tokenizer, prompt, 80, temperature=1.2, top_p=0.9))

---
## 8. 封装到 `llm/train.py` + `llm/generate.py`

In [ ]:
train_code = '''
import time
import torch
import torch.nn.functional as F


def calc_loss_batch(x, y, model, device):
    x, y = x.to(device), y.to(device)
    logits = model(x)
    return F.cross_entropy(logits.flatten(0, 1), y.flatten())


def calc_loss_loader(loader, model, device, num_batches=None):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if num_batches is not None and i >= num_batches:
                break
            total += calc_loss_batch(x, y, model, device).item()
            n += 1
    model.train()
    return total / max(n, 1)


def train(model, train_loader, val_loader, optimizer, device,
          num_epochs=1, eval_every=100, eval_batches=20,
          clip_grad=1.0, use_amp=True, on_eval=None):
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))
    history = {"step": [], "train_loss": [], "val_loss": []}
    step = 0
    t0 = time.time()
    model.train()
    for epoch in range(num_epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(use_amp and device == "cuda")):
                logits = model(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
            scaler.scale(loss).backward()
            if clip_grad is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            scaler.step(optimizer)
            scaler.update()
            step += 1
            if step % eval_every == 0:
                tr = calc_loss_loader(train_loader, model, device, eval_batches)
                vl = calc_loss_loader(val_loader, model, device, eval_batches)
                history["step"].append(step)
                history["train_loss"].append(tr)
                history["val_loss"].append(vl)
                print(f"[epoch {epoch+1} step {step:6d} {time.time()-t0:.0f}s] train={tr:.3f} val={vl:.3f}")
                if on_eval is not None:
                    on_eval(step, model)
    return history
'''.lstrip()

generate_code = '''
import torch


@torch.no_grad()
def generate(model, idx, max_new_tokens, context_length,
             temperature=1.0, top_k=None, top_p=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]
        logits = model(idx_cond)[:, -1, :]
        logits = logits / max(temperature, 1e-8)
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, -1:]] = float("-inf")
        probs = torch.softmax(logits, dim=-1)
        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cum = torch.cumsum(sorted_probs, dim=-1)
            mask = cum > top_p
            mask[..., 0] = False
            sorted_probs[mask] = 0.0
            sorted_probs = sorted_probs / sorted_probs.sum(-1, keepdim=True)
            next_id_local = torch.multinomial(sorted_probs, 1)
            next_id = sorted_idx.gather(-1, next_id_local)
        else:
            next_id = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_id], dim=1)
    return idx
'''.lstrip()

pathlib.Path('../llm/train.py').write_text(train_code, encoding='utf-8')
pathlib.Path('../llm/generate.py').write_text(generate_code, encoding='utf-8')
print('已写入 llm/train.py 和 llm/generate.py')

---
## 章末思考题

1. 为什么初始 loss 接近 ln(vocab_size)?模型还没学任何东西时,它在「预测」什么?
2. 混合精度(AMP)为什么能加速且省显存?会不会影响模型精度?
3. 温度、top-k、top-p 三种采样策略各适合什么场景?生成代码 vs 生成小说应该用哪种?

这章跑通后,你已经有一个能生成文本的 GPT 了。

**下一步(可选)**:第 6 章 —— 加载 OpenAI 官方 GPT-2 预训练权重,体验「质变」。